Database connection

In [35]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Load csv to dataframe

In [27]:
import pandas as pd
csv_file_path = "../dataset-fm2017/dataset-fm2017.csv"

df_csv = pd.read_csv(csv_file_path, dtype=str, encoding='utf-8-sig')
df_csv.columns = df_csv.columns.str.strip()

Load unique IDs from database

In [23]:
query = "SELECT unique_id FROM public.player_info;"
df_db = pd.read_sql(query, con=engine)

Compare IDs

In [32]:
csv_series = (
    df_csv['UID']
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(r'\.0$', '', regex=True)
)

db_series = (
    df_db['unique_id']
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(r'\.0$', '', regex=True)
)

csv_ids = set(csv_series)
db_ids = set(db_series)

# --- 5. Υπολογισμοί ---
matches = csv_ids & db_ids
only_in_csv = csv_ids - db_ids
only_in_db = db_ids - csv_ids

print(f"Μοναδικά IDs στο CSV:    {len(csv_ids):,}")
print(f"Μοναδικά IDs στη Βάση:   {len(db_ids):,}")
print(f"✅ Κοινά IDs (Matches):  {len(matches):,}")
print(f"❌ Μόνο στο CSV:         {len(only_in_csv):,}")
print(f"❌ Μόνο στη Βάση:        {len(only_in_db):,}")


Μοναδικά IDs στο CSV:    159,541
Μοναδικά IDs στη Βάση:   438,799
✅ Κοινά IDs (Matches):  64,352
❌ Μόνο στο CSV:         95,189
❌ Μόνο στη Βάση:        374,447


Δημιουργία dataframe που εχει μεσα τα σωστα columns

In [33]:
valid_ids = df_db['unique_id'].astype(str)
df_filtered = df_csv[df_csv['UID'].astype(str).isin(valid_ids)].copy()
df_filtered['year'] = 2017

# Αντιστοίχιση ονομάτων
rename_map = {
    'UID': 'unique_id', 'Corners': 'corners', 'Crossing': 'crossing', 'Dribbling': 'dribbling',
    'Finishing': 'finishing', 'FirstTouch': 'first_touch', 'Freekicks': 'free_kick_taking',
    'Heading': 'heading', 'LongShots': 'long_shots', 'Longthrows': 'long_throws',
    'Marking': 'marking', 'Passing': 'passing', 'PenaltyTaking': 'penalty_taking',
    'Tackling': 'tackling', 'Technique': 'technique', 'AerialAbility': 'aerial_reach',
    'CommandOfArea': 'command_of_area', 'Communication': 'communication',
    'Eccentricity': 'eccentricity', 'Handling': 'handling', 'Kicking': 'kicking',
    'OneOnOnes': 'one_on_ones', 'TendencyToPunch': 'punching', 'Reflexes': 'reflexes',
    'RushingOut': 'rushing_out', 'Throwing': 'throwing', 'Aggression': 'aggression',
    'Anticipation': 'anticipation', 'Bravery': 'bravery', 'Composure': 'composure',
    'Concentration': 'concentration', 'Decisions': 'decisions', 'Determination': 'determination',
    'Flair': 'flair', 'Leadership': 'leadership', 'OffTheBall': 'off_the_ball',
    'Positioning': 'positioning', 'Teamwork': 'teamwork', 'Vision': 'vision',
    'Workrate': 'work_rate', 'Acceleration': 'acceleration', 'Agility': 'agility',
    'Balance': 'balance', 'Jumping': 'jumping_reach', 'NaturalFitness': 'natural_fitness',
    'Pace': 'pace', 'Stamina': 'stamina', 'Strength': 'strength', 'Adaptability': 'adaptability',
    'Ambition': 'ambition', 'Consistency': 'consistency', 'Controversy': 'controversy',
    'Dirtiness': 'dirtiness', 'ImportantMatches': 'important_matches', 'InjuryProness': 'injury_proneness',
    'Loyalty': 'loyalty', 'Pressure': 'pressure', 'Professional': 'professionalism',
    'Sportsmanship': 'sportsmanship', 'Temperament': 'temperament', 'Versatility': 'versatility'
}

df_filtered = df_filtered.rename(columns=rename_map)

Import attributes to DB

In [36]:
from sqlalchemy.types import SmallInteger, VARCHAR

# Όλες οι τελικές στήλες που υπάρχουν στον πίνακα
db_columns = [
    'unique_id', 'year', 'corners', 'crossing', 'dribbling', 'finishing', 'first_touch',
    'free_kick_taking', 'heading', 'long_shots', 'long_throws', 'marking', 'passing',
    'penalty_taking', 'tackling', 'technique', 'aerial_reach', 'command_of_area',
    'communication', 'eccentricity', 'handling', 'kicking', 'one_on_ones', 'punching',
    'reflexes', 'rushing_out', 'throwing', 'aggression', 'anticipation', 'bravery',
    'composure', 'concentration', 'decisions', 'determination', 'flair', 'leadership',
    'off_the_ball', 'positioning', 'teamwork', 'vision', 'work_rate', 'acceleration',
    'agility', 'balance', 'jumping_reach', 'natural_fitness', 'pace', 'stamina',
    'strength', 'adaptability', 'ambition', 'consistency', 'controversy', 'dirtiness',
    'important_matches', 'injury_proneness', 'loyalty', 'pressure', 'professionalism',
    'sportsmanship', 'temperament', 'versatility'
]

df_final = df_filtered[db_columns].copy()

# 6. Μετατροπή στους ακριβείς τύπους μέσα στο DataFrame
# Το unique_id είναι string (character varying(50))
df_final['unique_id'] = df_final['unique_id'].astype(str)

# Όλα τα υπόλοιπα (και το year) είναι smallint (στην pandas 'Int16')
smallint_columns = [col for col in db_columns if col != 'unique_id']
for col in smallint_columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype('Int16')

# 7. Δημιουργία του λεξικού για τους τύπους της SQLAlchemy (ώστε να μιλήσει "1-προς-1" με τη Βάση)
sql_dtypes = {col: SmallInteger() for col in smallint_columns}
sql_dtypes['unique_id'] = VARCHAR(length=50)

# 8. Τελική Εισαγωγή
df_final.to_sql(
    'player_attributes',
    con=engine,
    schema='public',
    if_exists='append',
    index=False,
    dtype=sql_dtypes
)

print(f"Επιτυχής εισαγωγή {len(df_final)} γραμμών! Όλοι οι τύποι δεδομένων ήταν 100% αυστηροί.")

Επιτυχής εισαγωγή 64352 γραμμών! Όλοι οι τύποι δεδομένων ήταν 100% αυστηροί.
